<a href="https://colab.research.google.com/github/abdul4rehman215/AI-Advanced-Course-Portfolio/blob/main/04-openai-api-and-huggingface/Assignment_03_Abdul_Rehman.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Assignment 03 - OpenAI and Hugging Face

**Student Name:** Abdul Rehman  
**Course:** Generative AI & Natural Language Processing  
**Date:** October 2023  

---

## Assignment Overview
This notebook explores the practical application of Large Language Models (LLMs) through two primary ecosystems: **OpenAI** and **Hugging Face**. The tasks range from theoretical understanding of model architectures to building functional applications for sentiment analysis, Named Entity Recognition (NER), and content generation.

In [1]:
!pip install -q openai transformers torch accelerate

In [2]:
import os
import getpass
from openai import OpenAI
from transformers import pipeline, AutoTokenizer, AutoModelForSequenceClassification
import torch

print("Libraries imported successfully.")

Libraries imported successfully.


## Safe API Setup

In this section, we securely handle the OpenAI API key.

**Security Note:** It is critical never to hardcode API keys directly into source code. When using version control systems like GitHub, hardcoded keys can be scraped by malicious actors. Instead, we use `getpass` to input the key into the environment variables during the session only.

In [5]:
os.environ["OPENAI_API_KEY"] = getpass.getpass("Enter your OpenAI API Key: ")

client = None
try:
    if os.environ["OPENAI_API_KEY"]:
        client = OpenAI(api_key=os.environ["OPENAI_API_KEY"])
        print("OpenAI Client initialized successfully.")
    else:
        print("No API key detected. Please provide a key to run OpenAI-dependent cells.")
except Exception as e:
    print(f"Initialization failed: {e}")

Enter your OpenAI API Key: ··········
OpenAI Client initialized successfully.


## Question 1: Context Windows in LLMs

### What is a Context Window?
The context window is the maximum amount of text (measured in tokens) that a language model can "read" or consider at any one time before generating a response. Think of it as the model's short-term memory during a single conversation or task.

### Why is it typically limited?
Context windows are limited primarily due to computational costs. Most modern LLMs use the Transformer architecture, where the memory and processing power required grow quadratically with the length of the input. Processing extremely long sequences requires massive amounts of GPU RAM.

### OpenAI Model Families & Large Context
The **GPT-4** family (specifically `gpt-4-turbo` and `gpt-4o`) is known for significantly larger context windows (up to 128k tokens). This matters because it allows users to upload entire books, long legal documents, or massive codebases for analysis without the model "forgetting" the beginning of the text.

## Question 2: Challenges, Embeddings, and Fine-Tuning

### Challenges in Deployment
1. **Hallucination:** Models generating confident but false information.
2. **Bias:** Models reflecting societal prejudices found in training data.
3. **Inference Cost:** Running large models is expensive and energy-intensive.

### Addressing Challenges
Researchers use **RLHF (Reinforcement Learning from Human Feedback)** to align models with human values and **RAG (Retrieval-Augmented Generation)** to reduce hallucinations by providing the model with factual reference data.

### Understanding Embeddings
Embeddings are numerical representations of text (vectors). They convert words or sentences into a list of numbers where mathematically "close" vectors represent semantically similar meanings. This allows computers to understand relationships between concepts.

### Understanding Fine-Tuning
Fine-tuning is the process of taking a pre-trained model and training it further on a smaller, specific dataset. This adapts a general-purpose model for specialized tasks, such as medical diagnosis or specific brand voice mimicry.

In [6]:
def analyze_sentiment(text):
    if not client:
        return "Error: OpenAI Client not configured."

    try:
        response = client.chat.completions.create(
            model="gpt-3.5-turbo",
            messages=[
                {"role": "system", "content": "You are a sentiment analysis assistant. Classify the user text as Positive, Negative, or Neutral. Respond with only one word."},
                {"role": "user", "content": text}
            ]
        )
        return response.choices[0].message.content.strip()
    except Exception as e:
        return f"API Error: {e}"

# Testing
test_samples = [
    "I absolutely love the new design of this app!",
    "The food was okay, but the service was quite slow.",
    "I am so frustrated with this constant crashing."
]

print("--- Sentiment Analysis Results ---")
for sample in test_samples:
    print(f"Text: {sample}\nSentiment: {analyze_sentiment(sample)}\n")

--- Sentiment Analysis Results ---
Text: I absolutely love the new design of this app!
Sentiment: Positive

Text: The food was okay, but the service was quite slow.
Sentiment: Negative

Text: I am so frustrated with this constant crashing.
Sentiment: Negative



In [7]:
def calculate_similarity(text1, text2):
    if not text1 or not text2:
        return "Error: Inputs cannot be empty."
    if not client:
        return "Error: OpenAI Client not configured."

    try:
        prompt = f"Compare the following two texts and provide a similarity score from 0 to 100, where 100 means identical meaning and 0 means completely unrelated. Provide only the number.\n\nText 1: {text1}\nText 2: {text2}"
        response = client.chat.completions.create(
            model="gpt-3.5-turbo",
            messages=[{"role": "user", "content": prompt}]
        )
        score = response.choices[0].message.content.strip()
        return f"Similarity Score: {score}/100"
    except Exception as e:
        return f"Failed to calculate similarity: {e}"

t1 = "The weather is sunny today."
t2 = "It is a bright and clear day outside."
print(calculate_similarity(t1, t2))

Similarity Score: 90/100


In [8]:
def generate_blog_package(topic):
    if not client: return "API Key missing."

    try:
        prompt = f"""Generate a blog post package for the topic: '{topic}'.
        Include:
        1. Title
        2. 300-word Body
        3. Keywords
        4. SEO Meta Title
        5. SEO Meta Description
        6. A descriptive prompt for an AI image generator representing this blog.
        Format each section clearly."""

        response = client.chat.completions.create(
            model="gpt-3.5-turbo",
            messages=[{"role": "user", "content": prompt}]
        )
        return response.choices[0].message.content
    except Exception as e:
        return f"An error occurred: {e}"

user_topic = input("Enter a blog topic: ")
if user_topic:
    print("\n--- Generated Blog Content ---\n")
    print(generate_blog_package(user_topic))
else:
    print("No topic provided.")

Enter a blog topic: Water Benefits in Accordance to Quran and Science

--- Generated Blog Content ---

Title: The Healing Power of Water: Insights from the Quran and Science

300-word Body:
Water is considered a vital element in both religious scriptures and scientific studies, with the Quran promoting its various benefits for physical and spiritual well-being. In the holy book, water is often mentioned as a source of life and purification, emphasizing its importance for sustenance and cleansing. From performing ablution before prayer to seeking blessings through the Zamzam well, water holds a special place in Islamic teachings.

On the scientific front, research has shown the numerous health benefits of water, including maintaining hydration, regulating body temperature, aiding digestion, and flushing out toxins. The human body is composed of around 60% water, highlighting its essential role in overall health and functioning. Studies have also revealed the positive impact of water on 

## Question 6: Understanding Hugging Face

### What is Hugging Face?
Hugging Face is often called the "GitHub of AI." It is a massive community and platform that hosts open-source models, datasets, and demo apps (Spaces), making AI accessible to everyone.

### Why is it important?
It democratizes AI by allowing developers to use state-of-the-art models (like BERT, Llama, or Stable Diffusion) without needing the massive resources required to train them from scratch.

### Transformers and Pipelines
- **Transformers:** The specific neural network architecture that handles sequential data (like text) efficiently using 'Attention' mechanisms.
- **Pipelines:** A high-level helper function in the Hugging Face library that abstracts away complex code, allowing you to perform tasks like sentiment analysis or translation in just two lines of code.

### Hugging Face API
While many use the library locally, the **Inference API** allows you to run models hosted on Hugging Face servers via HTTP requests, similar to how we use OpenAI.

In [9]:
def hf_sentiment_analysis(text_list):
    # Using a default pre-trained model for sentiment
    classifier = pipeline("sentiment-analysis", model="distilbert-base-uncased-finetuned-sst-2-english")
    results = classifier(text_list)

    for text, res in zip(text_list, results):
        print(f"Text: {text}")
        print(f"Label: {res['label']}, Score: {res['score']:.4f}\n")

samples = [
    "This assignment is very informative!",
    "I am struggling with the complex math involved.",
    "The sun rises in the east."
]

hf_sentiment_analysis(samples)

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

Text: This assignment is very informative!
Label: POSITIVE, Score: 0.9998

Text: I am struggling with the complex math involved.
Label: NEGATIVE, Score: 0.9989

Text: The sun rises in the east.
Label: POSITIVE, Score: 0.9991



In [11]:
# Updated to use 'aggregation_strategy' instead of the deprecated 'grouped_entities'
ner_pipeline = pipeline("ner", aggregation_strategy="simple")

corpus = """Apple Inc. is looking at buying a startup in San Francisco.
Meanwhile, Microsoft expanded its headquarters in Redmond, Washington last year."""

results = ner_pipeline(corpus)

print("--- Named Entity Extraction ---")
print("Companies Found:")
for ent in results:
    # Using 'entity_group' which is populated when aggregation_strategy is used
    if ent['entity_group'] == 'ORG':
        print(f"- {ent['word']}")

print("\nLocations Found:")
for ent in results:
    if ent['entity_group'] == 'LOC':
        print(f"- {ent['word']}")

No model was supplied, defaulted to dbmdz/bert-large-cased-finetuned-conll03-english and revision 4c53496.
Using a pipeline without specifying a model name and revision in production is not recommended.


Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

BertForTokenClassification LOAD REPORT from: dbmdz/bert-large-cased-finetuned-conll03-english
Key                      | Status     |  | 
-------------------------+------------+--+-
bert.pooler.dense.weight | UNEXPECTED |  | 
bert.pooler.dense.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


--- Named Entity Extraction ---
Companies Found:
- Apple Inc
- Microsoft

Locations Found:
- San Francisco
- Redmond
- Washington


In [12]:
generator = pipeline('text-generation', model='gpt2')

def creative_continuation(prompt, max_len=50):
    output = generator(prompt, max_length=max_len, num_return_sequences=1, truncation=True)
    return output[0]['generated_text']

prompts = [
    "In a galaxy far away, there lived a",
    "The secret to learning AI quickly is",
    "When the robot finally woke up, it said"
]

for p in prompts:
    print(f"Prompt: {p}")
    print(f"Result: {creative_continuation(p)}\n" + "-"*30)

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: gpt2
Key                  | Status     |  | 
---------------------+------------+--+-
h.{0...11}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Passing `generation_config` together with generation-related arguments=({'max_length', 'num_return_sequences'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Both `max_new_tokens` (=256) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Prompt: In a galaxy far away, there lived a


Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Both `max_new_tokens` (=256) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Result: In a galaxy far away, there lived a kind of alien race called Korran, who were not unlike a human race, but were much more alien. Korran's name means "kind", "goddess of the heavens", "god of the sky", "god of the clouds", "god of the moon", and "god of the stars". In order to be born into Korran's race, one must have something of an astral soul.

In the game, the human race was created by the planet Orion, and their race was created by the galaxy's stars. When they were made to be able to manifest their soul, they were created in the shape of a human.

A human born star will make an alien of one of the other planets.

One of the planets in the game is called the Delta Quadrant, but also a planet called the Galaxy. The planets are all in the Delta Quadrant, and are all called the Delta Quadrant by the player.

In the game, the player can travel to the planet and collect the planet's resources, which can then be used to build a super-star that will create their new human race.



Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Both `max_new_tokens` (=256) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Result: The secret to learning AI quickly is in its ability to understand the environment, and to learn to navigate to a different location.

"This is all part of the experience," said Professor Heng, who is a professor at the University of California, San Francisco and has been studying AI for more than three decades. "It's not just about learning, it's about understanding."

The new system will be built with the help of a team of researchers from the University of California, Berkeley, and the University of California, Santa Cruz. The system will be able to navigate a virtual reality environment by moving objects from one virtual state to another.

The researchers hope the system will help them understand the emotional state of a person, and to learn how to recognize a person's emotion.

"The brain is designed to be flexible, flexible and flexible," said Steven K. Sperry, a professor of psychology at the University of Florida. "You can't just jump from one virtual state to another."


In [13]:
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")
sentence = "Tokenization is the first step in NLP."

inputs = tokenizer(sentence)
tokens = tokenizer.tokenize(sentence)

print(f"Original: {sentence}")
print(f"Tokens: {tokens}")
print(f"Token IDs: {inputs['input_ids']}")
print(f"Attention Mask: {inputs['attention_mask']}")

Original: Tokenization is the first step in NLP.
Tokens: ['token', '##ization', 'is', 'the', 'first', 'step', 'in', 'nl', '##p', '.']
Token IDs: [101, 19204, 3989, 2003, 1996, 2034, 3357, 1999, 17953, 2361, 1012, 102]
Attention Mask: [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]


## What I Practiced in OpenAI and Hugging Face
- **API Integration:** Learned how to safely manage and call cloud-based models.
- **Prompt Engineering:** Experimented with system and user roles to get specific output formats.
- **Pipeline Abstraction:** Used Hugging Face pipelines to quickly deploy NER and Sentiment models.
- **Tokenization Mechanics:** Visualized how text is broken down for machine consumption.

## Challenges Faced and Notes for Future Improvement
- **Rate Limiting:** Managing API usage to avoid hitting trial limits.
- **Model Selection:** Understanding that smaller models (like GPT2) are faster but less coherent than larger ones (GPT-3.5).
- **Entity Grouping:** Realizing that NER models often split names (like 'Apple Inc.') into multiple tokens unless properly grouped.